# 01 · Few-Label Land Use (contest exercise)

**Style C: contest simulation** in the IOAI 2026 format: an **allow-listed** pretrained model, a short grade-time
budget that must cover training, and a public split for self-scoring plus two hidden leaderboards.
**Traces to** `notes/ioai-task-patterns.md` P1, P2, P3, P4, P7.

> **Reference solution.** Fine-tune the allowed ResNet18 at 128 px with strong but label-preserving augmentation
> (all flips and 90° rotations, which are valid for satellite tiles, plus crops and colour jitter) and label smoothing.
> Use a cosine schedule and 8-view dihedral test-time augmentation.

## 1. Problem Description

A Sentinel-2 monitoring project needs a land-use classifier for 64×64 satellite tiles, but experts have labelled
only **30 tiles per class**. Unlabelled tiles are cheap, labels are not.

## 2. Dataset (EuroSAT RGB, loaded below)

| Split | Tiles | Labels |
|---|---|---|
| `train` | 300 (30 per class) | ✅ |
| `test_a` | 1,500 | hidden (Leaderboard A) |
| `test_b` | 1,500 | hidden (Leaderboard B, **final**) |

Classes: Annual Crop, Forest, Herbaceous Vegetation, Highway, Industrial Buildings, Pasture, Permanent Crop,
Residential Buildings, River, SeaLake.

## 3. Task

Predict the class id (0–9) of every test tile.

## 4. Submission

`submission.zip` containing `submission_a.csv` and `submission_b.csv`, each **with header** `id,label` and one row
per test tile (`id` = position 0..1499).

## 5. Scoring

Accuracy.
- Baseline (Leaderboard B): **0.53** (a small CNN trained from scratch, below)
- Reference (Leaderboard B): **0.93** (in about 1 minute on an RTX 3060), which is your target

## 6. Requirements

- **Allowed pretrained models:** `torchvision.models.resnet18` with `ResNet18_Weights.DEFAULT` only.
- The whole notebook, training included, runs in **≤ 10 minutes on a T4**.
- No external data. The test tiles may not be used for training (they are hidden on the real platform).

In [ ]:
import time, os, io, zipfile, random
T0 = time.time()
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models
from torchvision.transforms import v2
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"

### Data loading (DO NOT CHANGE)

In [ ]:
from datasets import load_dataset
def _load():
    ds = load_dataset("blanchon/EuroSAT_RGB")
    rng = np.random.default_rng(2026)
    lab_tr, lab_te = np.array(ds["train"]["label"]), np.array(ds["test"]["label"])
    i_tr = np.concatenate([rng.choice(np.flatnonzero(lab_tr == c), 30, replace=False) for c in range(10)])
    i_te = rng.permutation(len(lab_te))[:3000]
    to_u8 = lambda split, idx: torch.from_numpy(np.stack([np.array(split[int(i)]["image"].convert("RGB")) for i in idx])).permute(0, 3, 1, 2)
    return to_u8(ds["train"], i_tr), torch.tensor(lab_tr[i_tr]), to_u8(ds["test"], i_te), lab_te[i_te]

X_train, y_train, _X_te, _y_te = _load()          # uint8 (N, 3, 64, 64)
X_test_a, X_test_b = _X_te[:1500], _X_te[1500:]
_ANSWERS = (_y_te[:1500], _y_te[1500:])            # the organisers' copy: do not use
del _X_te, _y_te
print(X_train.shape, X_train.dtype, X_test_a.shape)

### Reference solution

In [ ]:
W = models.ResNet18_Weights.DEFAULT
MEAN = torch.tensor(W.transforms().mean, device=device).view(1, 3, 1, 1)
STD = torch.tensor(W.transforms().std, device=device).view(1, 3, 1, 1)
SIZE = 128

def prep(x_u8):                                            # uint8 batch on device -> normalised float
    x = F.interpolate(x_u8.float() / 255, size=SIZE, mode="bilinear", align_corners=False, antialias=True)
    return (x - MEAN) / STD

def dihedral(x, k):                                         # the 8 symmetries of a square
    x = torch.rot90(x, k % 4, dims=(2, 3))
    return x.flip(3) if k >= 4 else x

aug = v2.Compose([v2.RandomResizedCrop(SIZE, scale=(0.5, 1.0), antialias=True),
                  v2.ColorJitter(0.3, 0.3, 0.2, 0.02)])

def build_and_predict(X_train, y_train, tests, epochs=30, bs=32):
    model = models.resnet18(weights=W); model.fc = nn.Linear(512, 10); model = model.to(device)
    X = X_train.to(device); y = y_train.to(device)
    steps = epochs * ((len(X) + bs - 1) // bs)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=steps, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    for ep in range(epochs):
        model.train()
        for idx in torch.randperm(len(X), device=device).split(bs):
            xb = torch.stack([aug(dihedral(x[None], random.randrange(8))[0]) for x in X[idx]])
            xb = (xb.float() / 255 - MEAN) / STD
            with torch.autocast("cuda", dtype=torch.float16, enabled=device == "cuda"):
                loss = F.cross_entropy(model(xb), y[idx], label_smoothing=0.1)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    model.eval(); outs = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=device == "cuda"):
        for t in tests:
            preds = []
            for i in range(0, len(t), 250):
                xb = prep(t[i:i + 250].to(device))
                probs = sum(model(dihedral(xb, k)).float().softmax(1) for k in range(8))
                preds.append(probs.argmax(1).cpu())
            outs.append(torch.cat(preds).numpy())
    return outs

### Local validation (5-fold would be better; one split keeps it quick)

In [ ]:
perm = torch.randperm(len(X_train))
va, tr = perm[:60], perm[60:]
(pv,) = build_and_predict(X_train[tr], y_train[tr], [X_train[va]])
print(f"local accuracy on 60 held-out tiles: {(pv == y_train[va].numpy()).mean():.3f}   ({time.time() - T0:.0f}s elapsed)")

### Submission

In [ ]:
pred_a, pred_b = build_and_predict(X_train, y_train, [X_test_a, X_test_b])
with zipfile.ZipFile("submission.zip", "w") as z:
    for name, p in [("submission_a.csv", pred_a), ("submission_b.csv", pred_b)]:
        z.writestr(name, pd.DataFrame({"id": np.arange(len(p)), "label": p.astype(int)}).to_csv(index=False))
print("wrote submission.zip")

### Official grader (run it, don't read it)

In [ ]:
SHOW_FINAL = False or os.environ.get("IOAI_SHOW_FINAL") == "1"

def _grade(zip_path="submission.zip"):
    out = []
    with zipfile.ZipFile(zip_path) as z:
        for name, truth in zip(["submission_a.csv", "submission_b.csv"], _ANSWERS):
            try:
                df = pd.read_csv(z.open(name))
                assert list(df.columns) == ["id", "label"] and (df["id"].to_numpy() == np.arange(len(truth))).all()
                out.append(float((df["label"].to_numpy() == truth).mean()))
            except Exception as e:
                print(name, "rejected:", e); out.append(0.0)
    return out

score_a, score_b = _grade()
print(f"Leaderboard A (public): {score_a:.4f}")
print(f"Leaderboard B (final):  {score_b:.4f}" if SHOW_FINAL else "Leaderboard B: hidden until SHOW_FINAL = True")
elapsed = time.time() - T0
print(f"total runtime {elapsed / 60:.1f} min (limit 10 min)")
assert elapsed < 600, "over the time limit: on the platform this run would score 0"